# This is a working version on Google Colab. To run the code, it is suggested to import and run on Google Colab. Some code blocks may need to be uncommented

In [ ]:
# RUN THIS FIRST

# Needs Token from Hugging Face
# from huggingface_hub import login
# login()

ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.

In [ ]:
# !nvidia-smi

# If does not work, change runtime settings by going to Runtime in the top bar -> Change runtime type -> Hardware Accelerator -> A100

Fri Feb 27 19:07:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Install vLLM

#!pip -q install -U transformers accelerate bitsandbytes sentencepiece pandas openpyxl jsonschema

#!pip -q install -U "transformers>=4.41.0" "accelerate>=0.31.0" "bitsandbytes>=0.43.1" sentencepiece safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 147.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires panda

In [ ]:
# Upload labels.xlsx to Colab first

import pandas as pd

df = pd.read_excel("labels.xlsx")
print(df.shape)
print(df.columns)
df.head(3)

(6, 2)
Index(['img_file', 'labels'], dtype='str')


,img_file,labels
0,house1.jpeg,"pool: yes, enclosed_pool: no, spa: yes, roof_..."
1,bike1.jpg,"brand: saffron, color: green, condition: new, ..."
2,boat1.jpeg,"brand: sea-doo, color: white, condition: used,..."


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

print("Loaded OK")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded OK


In [ ]:
import json
from jsonschema import validate, ValidationError

SCHEMA = {
  "type": "object",
  "properties": {
    "img_file": {"type": "string"},
    "labels": {
      "type": "object",
      "additionalProperties": {"type": "string"}
    }
  },
  "required": ["img_file", "labels"],
  "additionalProperties": False
}

def validate_json(obj):
    validate(instance=obj, schema=SCHEMA)
    return True

In [ ]:
def qwen_generate(system_prompt, user_prompt, max_new_tokens=200, temperature=0.1):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    input_len = inputs["input_ids"].shape[1]
    print(f"DEBUG: max_new_tokens={max_new_tokens}, input_tokens={input_len}, total={input_len + max_new_tokens}")

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    new_tokens = out[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


SYSTEM_PARSE = (
    "Output ONLY a raw JSON object. "
    "No markdown. No code fences. No backticks. No explanation. "
    "Start your response with { and end with }.\n"
    'Schema: { "img_file": string, "labels": { string: string } }\n'
    "Keep all keys exactly as given. All values must be strings."
)

REPAIR_SYSTEM = (
    "You are a JSON repair tool. Output ONLY valid JSON. No other text."
)

import re

def extract_first_json(text):
    text = re.sub(r"```(?:json)?", "", text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in:\n{text[:300]}")
    return text[start:end+1]

def parse_row_with_repair(img_file, labels_raw):
    user = f"Convert into valid JSON.\nimg_file={img_file}\nlabels={labels_raw}\n"
    raw = qwen_generate(SYSTEM_PARSE, user)

    print("=== RAW OUTPUT ===")
    print(repr(raw[:600]))
    print("==================")

    try:
        candidate = extract_first_json(raw)
        obj = json.loads(candidate)
        validate_json(obj)
        return obj, raw, True
    except Exception as e:
        print(f"First pass failed: {e}")
        repair_user = (
            "Fix this into valid JSON matching the schema exactly:\n"
            f"{raw}\n"
        )
        repaired = qwen_generate(REPAIR_SYSTEM, repair_user, temperature=0.0)
        candidate = extract_first_json(repaired)
        obj = json.loads(candidate)
        validate_json(obj)
        return obj, repaired, False

In [ ]:
results = []
logs = []

for _, row in df.head(10).iterrows():
    img_file = str(row["img_file"])
    labels_raw = str(row["labels"])

    obj, raw_text, first_pass_ok = parse_row_with_repair(img_file, labels_raw)
    results.append(obj)
    logs.append({
        "img_file": img_file,
        "first_pass_ok": first_pass_ok,
        "raw_output_preview": raw_text[:500]
    })

print("Parsed:", len(results))
print("First sample:", results[0])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


DEBUG: max_new_tokens=200, input_tokens=142, total=342
=== RAW OUTPUT ===
'{"img_file":"house1.jpeg","labels":{"pool":"yes","enclosed_pool":"no","spa":"yes","roof_discoloration":"yes","roof_damage":"no","roof_type":"shingles","trampoline":"no","solar_panels":"no","approx_sqft":"?","approx_value":"?"}}'
DEBUG: max_new_tokens=200, input_tokens=112, total=312
=== RAW OUTPUT ===
'{"img_file": "bike1.jpg", "labels": {"brand": "saffron", "color": "green", "condition": "new", "style": "ten-speed", "approx_value": "?"}}'
DEBUG: max_new_tokens=200, input_tokens=119, total=319
=== RAW OUTPUT ===
'{"img_file":"boat1.jpeg","labels":{"brand":"sea-doo","color":"white","condition":"used","style":"4-seat_sport","trailer":"yes","approx_value":"?"}}'
DEBUG: max_new_tokens=200, input_tokens=128, total=328
=== RAW OUTPUT ===
'{"img_file":"car1.jpeg","labels":{"brand":"bmw","color":"red","condition":"new","model":"bmw_series_2_gran_coupe","style":"coupe","sun_roof":"yes","approx_value":"?"}}'
DEBUG: max_ne